In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix


SEED = 42
TEST_SIZE = 0.2
DATA = "../data/train.csv"
DROP = ["PassengerId", "Name", "Ticket", "Cabin", "SibSp"]

df = pd.read_csv(DATA).drop(columns=DROP)
X, y = df.drop(columns="Survived"), df["Survived"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)

In [2]:
num_cols = ["Age", "Fare", "Parch", "Pclass"]
cat_cols = ["Sex", "Embarked"]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore", drop="first")),
])
preprocess = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("cat", categorical_pipe, cat_cols),
])


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)

model = Pipeline([
    ("prep", preprocess),
    ("clf",  LogisticRegression(max_iter=1000, random_state=SEED)),
])
model.fit(X_train, y_train)

y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, digits=3))
print("ROC-AUC:", round(roc_auc_score(y_test, y_proba), 3))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0      0.802     0.845     0.823       110
           1      0.730     0.667     0.697        69

    accuracy                          0.777       179
   macro avg      0.766     0.756     0.760       179
weighted avg      0.774     0.777     0.774       179

ROC-AUC: 0.829
[[93 17]
 [23 46]]


In [3]:
import sys, os
sys.path.append(os.path.abspath("../.."))   
from utils import log_result

log_result(
    model_name="LogReg",
    y_true=y_test,
    y_proba=y_proba,      
    y_pred=y_pred,        
    params={"numeric_pipe": "median", "categorical_pipe": "most_frequent + OneHotEncoder"},
    notes=""
)

✓ LogReg: AUC=0.829, acc=0.7765 → ../results.csv


{'model': 'LogReg',
 'timestamp': '2026-11-04 01:09',
 'auc': 0.829,
 'accuracy': 0.7765,
 'f1': 0.697,
 'precision': 0.7302,
 'recall': 0.6667,
 'params': "{'numeric_pipe': 'median', 'categorical_pipe': 'most_frequent + OneHotEncoder'}",
 'notes': ''}